# A1.10 · Bit manipulation and a complexity-analysis self-quiz

**ID:** A1.10 · **Target:** 60 min · **Plan week:** 7 (pattern notebook, Mon); complexity analysis is used in every A1 notebook; timed mixed sets in week 16 · **Part:** A

**The drill.** This is a 60-minute study notebook, in two halves. **Bits (30 min):** fill in nine short functions:
Kernighan's popcount, power-of-two and lowest-set-bit tests, XOR tricks for "single" and "missing" numbers, a
counting-bits DP, enumerating subsets with masks, reversing 32 bits, and adding 32-bit signed integers without `+`
(Python's unbounded integers make this the trickiest one). **Complexity (30 min):** measure amortized list growth and
an $O(n)$ vs $O(n^2)$ queue, fill in a log-log slope estimator, then answer a 12-question self-quiz (recurrences,
the master theorem, amortized analysis, Python's hidden costs). Answers are in the solution notebook.

**Rules.** Python standard library only. Do not use `bin()`, `int.bit_count()` or `+`/`-` where a problem forbids
them; the tests use them as references. Fill in every `raise NotImplementedError`.

## Sources
- Python docs, *Bitwise operations on integer types* (results "as though carried out in two's complement with an
  infinite number of sign bits"), `int.bit_count()` (Python ≥ 3.10), `int.bit_length()`:
  https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types
- S. E. Anderson, *Bit Twiddling Hacks* (Stanford), "Counting bits set, Brian Kernighan's way", "Determining if an
  integer is a power of 2": https://graphics.stanford.edu/~seander/bithacks.html
- Tech Interview Handbook, *Binary* cheatsheet (masks, `x & (x - 1)`, negative numbers in Python):
  https://www.techinterviewhandbook.org/algorithms/binary/ (yangshun/tech-interview-handbook @ e1d28e8, MIT).
- T. H. Cormen, C. E. Leiserson, R. L. Rivest, C. Stein, *Introduction to Algorithms*, 4th ed., MIT Press, 2022
  (ISBN 9780262046305): Ch. 3 "Characterizing Running Times" ($O$, $\Omega$, $\Theta$), Ch. 4 "Divide-and-Conquer"
  (recursion trees, the master method), Ch. 16 "Amortized Analysis" (aggregate, accounting, potential; dynamic tables).
  https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/
- R. E. Tarjan, *Amortized Computational Complexity*, SIAM J. Algebraic Discrete Methods 6(2), 1985, DOI 10.1137/0606031.
- *Master theorem (analysis of algorithms)*, Wikipedia: https://en.wikipedia.org/wiki/Master_theorem_(analysis_of_algorithms)
- CPython 3.11 `Objects/listobject.c`, `list_resize` (over-allocation `newsize + (newsize >> 3) + 6`, growth pattern
  0, 4, 8, 16, 24, 32, 40, 52, 64, 76, ...): https://github.com/python/cpython/blob/3.11/Objects/listobject.c ;
  Python operation costs: https://wiki.python.org/moin/TimeComplexity
- NeetCode 150 problem data (practice list): https://neetcode.io/practice ,
  https://github.com/neetcode-gh/leetcode @ d5ebb3e `.problemSiteData.json`.

In [1]:
import contextlib
import copy
import math
import random
import signal
import struct
import sys
import time
from collections import Counter, deque

from prep import Session, check

s = Session("A1.10", target_min=60)
random.seed(0)


def run_tests(fn, cases, name=None):
    """cases: list of (args_tuple, expected). Stops at the first failing case and prints it."""
    name = name or fn.__name__
    for i, (args, want) in enumerate(cases):
        got = fn(*copy.deepcopy(args))
        assert got == want, f"{name} case {i}: args={args!r}\n  got  {got!r}\n  want {want!r}"
    print(f"ok  {name}: {len(cases)} cases")


def stress(fn, brute, gen, n=300, seed=0, name=None):
    """Compare fn against a trusted brute force on n random small inputs; gen(rng) returns an args tuple."""
    name = name or fn.__name__
    rng = random.Random(seed)
    for t in range(n):
        args = gen(rng)
        want = brute(*copy.deepcopy(args))
        got = fn(*copy.deepcopy(args))
        assert got == want, f"{name} vs brute force, trial {t}: args={args!r}\n  got  {got!r}\n  want {want!r}"
    print(f"ok  {name} vs brute force: {n} random cases")


@contextlib.contextmanager
def time_limit(sec=2):
    """Raise TimeoutError if the block runs longer than sec (an unmasked negative number never stops carrying). Unix only."""
    if not hasattr(signal, "SIGALRM"):
        yield
        return

    def handler(signum, frame):
        raise TimeoutError(f"took more than {sec}s: infinite carry loop? (mask both operands with 0xFFFFFFFF)")

    old = signal.signal(signal.SIGALRM, handler)
    signal.setitimer(signal.ITIMER_REAL, sec)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)

[A1.10] started 12:41; target 60 min


## 1. Bit tricks
**Intuition.** `x & (x - 1)` clears the **lowest set bit**: subtracting 1 flips that bit and every 0 below it.
`x & -x` isolates the lowest set bit, because `-x = ~x + 1` in two's complement. XOR is associative and commutative,
with `a ^ a = 0` and `a ^ 0 = a`, so XOR-ing a multiset cancels every value that appears an even number of times. A
mask `m` with $n$ bits is a subset of $n$ items.

**Signals:** "single/missing/duplicate number in $O(1)$ space", "power of two", "count bits", "all subsets with
$n \le 20$" (masks; bitmask DP), "without using + or *", flags packed in an integer, Gray codes.

**Python caveat.** `int`s are unbounded and behave like two's complement with **infinitely many sign bits** (Python
docs). So `~5 == -6`, and `-1 >> 1 == -1`. For 32-bit semantics, mask with `0xFFFFFFFF` and convert back to signed at the end.

In [2]:
def popcount(x):
    """Number of 1 bits of x >= 0, by clearing the lowest set bit until 0 (Kernighan). O(popcount)."""
    c = 0
    while x:
        x &= x - 1
        c += 1
    return c


def is_power_of_two(x):
    return x > 0 and x & (x - 1) == 0


def lowest_set_bit(x):
    """The value of the lowest set bit of x > 0 (e.g. 12 -> 4)."""
    return x & -x


run_tests(popcount, [((0,), 0), ((11,), 3), ((2**40 - 1,), 40), ((128,), 1)])
stress(popcount, lambda x: bin(x).count("1"), lambda r: (r.getrandbits(r.randint(0, 70)),), n=1000)
run_tests(is_power_of_two, [((1,), True), ((0,), False), ((-8,), False), ((6,), False), ((2**60,), True)])
stress(is_power_of_two, lambda x: x > 0 and bin(x).count("1") == 1, lambda r: (r.randint(-70, 1100),), n=2000)
run_tests(lowest_set_bit, [((12,), 4), ((1,), 1), ((96,), 32)])
stress(lowest_set_bit, lambda x: next(1 << i for i in range(x.bit_length()) if x >> i & 1),
       lambda r: (r.randint(1, 10**6),), n=1000)

ok  popcount: 4 cases
ok  popcount vs brute force: 1000 random cases
ok  is_power_of_two: 5 cases
ok  is_power_of_two vs brute force: 2000 random cases
ok  lowest_set_bit: 3 cases
ok  lowest_set_bit vs brute force: 1000 random cases


### Warm-ups 1.1-1.2: XOR tricks (~6 min)
(a) Every value appears exactly twice except one, which appears once: return it in $O(n)$ time and $O(1)$ space.
(b) `nums` holds $n$ distinct values from `0..n`, so exactly one is missing: return it without sorting.
XOR the indices with the values.

In [3]:
def single_number(nums):
    x = 0
    for v in nums:
        x ^= v
    return x


def missing_number(nums):
    x = len(nums)
    for i, v in enumerate(nums):
        x ^= i ^ v
    return x


def _gen_single(r):
    vals = r.sample(range(-20, 20), r.randint(1, 6))
    nums = vals[1:] * 2 + vals[:1]
    r.shuffle(nums)
    return (nums,)


def _gen_missing(r):
    n = r.randint(0, 9)
    nums = list(range(n + 1))
    nums.remove(r.randint(0, n))
    r.shuffle(nums)
    return (nums,)


run_tests(single_number, [(([4, 1, 2, 1, 2],), 4), (([-3],), -3), (([0, 7, 7],), 0)])
stress(single_number, lambda nums: next(v for v, c in Counter(nums).items() if c == 1), _gen_single)
run_tests(missing_number, [(([3, 0, 1],), 2), (([],), 0), (([0],), 1), (([1],), 0)])
stress(missing_number, lambda nums: next(i for i in range(len(nums) + 1) if i not in nums), _gen_missing)

ok  single_number: 3 cases
ok  single_number vs brute force: 300 random cases
ok  missing_number: 4 cases
ok  missing_number vs brute force: 300 random cases


### Warm-up 1.3: counting bits for 0..n in $O(n)$ (~5 min)
Return `[popcount(0), ..., popcount(n)]` without a per-number loop. *Hint:* `i >> 1` has already been computed,
and `i & 1` is the bit you dropped. (Alternatively, `bits[i & (i - 1)] + 1`.)

In [4]:
def counting_bits(n):
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits


run_tests(counting_bits, [((0,), [0]), ((5,), [0, 1, 1, 2, 1, 2])])
stress(counting_bits, lambda n: [bin(i).count("1") for i in range(n + 1)], lambda r: (r.randint(0, 300),), n=100)

ok  counting_bits: 2 cases
ok  counting_bits vs brute force: 100 random cases


### Warm-up 1.4: all subsets via masks (~5 min)
Return every subset of `nums` (distinct values), with mask `m` mapping to `[nums[i] for each set bit i]`. The order
of subsets does not matter (the test sorts them). Compare with the backtracking version in A1.8.

In [5]:
def subsets_by_mask(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if m >> i & 1] for m in range(1 << n)]


def _subsets_ref(nums):
    out = [[]]
    for x in nums:
        out += [s_ + [x] for s_ in out]
    return out


canon = lambda f: (lambda nums: sorted(sorted(x) for x in f(nums)))
run_tests(canon(subsets_by_mask), [(([1, 2],), [[], [1], [1, 2], [2]]), (([],), [[]])], name="subsets_by_mask")
stress(canon(subsets_by_mask), canon(_subsets_ref), lambda r: (r.sample(range(10), r.randint(0, 6)),), name="subsets_by_mask")

ok  subsets_by_mask: 2 cases
ok  subsets_by_mask vs brute force: 300 random cases


### Warm-up 1.5: reverse the 32 bits of an unsigned integer (~5 min)
Bit `i` of the input becomes bit `31 - i` of the output, for `0 <= x < 2**32`.

In [6]:
def reverse_bits_32(x):
    out = 0
    for _ in range(32):
        out = (out << 1) | (x & 1)
        x >>= 1
    return out


run_tests(reverse_bits_32, [((0b00000010100101000001111010011100,), 0b00111001011110000010100101000000),
                            ((1,), 2**31), ((0,), 0), ((2**32 - 1,), 2**32 - 1)])
stress(reverse_bits_32, lambda x: int(format(x, "032b")[::-1], 2), lambda r: (r.getrandbits(32),), n=1000)

ok  reverse_bits_32: 4 cases
ok  reverse_bits_32 vs brute force: 1000 random cases


### Warm-up 1.6: add two 32-bit signed integers without `+` or `-` (~10 min)
`a ^ b` is the sum without carries, and `(a & b) << 1` is the carries. Repeat until there is no carry. In Python,
keep both values masked to 32 bits (`& 0xFFFFFFFF`), or a negative operand loops forever because its sign bits never
run out. Convert the result back to signed: if bit 31 is set, the value is `~(r ^ 0xFFFFFFFF)`. Inputs and the true
sum fit in a signed 32-bit int.

In [7]:
def add_32(a, b):
    MASK, MAX = 0xFFFFFFFF, 0x7FFFFFFF
    a, b = a & MASK, b & MASK
    while b:
        a, b = (a ^ b) & MASK, ((a & b) << 1) & MASK
    return a if a <= MAX else ~(a ^ MASK)


def _gen_add(r):
    a = r.randint(-2**31, 2**31 - 1)
    b = r.randint(max(-2**31, -2**31 - a), min(2**31 - 1, 2**31 - 1 - a))
    return a, b


with time_limit():
    run_tests(add_32, [((1, 2), 3), ((-1, 1), 0), ((-5, -7), -12), ((2**31 - 1, 0), 2**31 - 1), ((-2**31, 0), -2**31),
                       ((-2**31 + 5, -5), -2**31)])
    stress(add_32, lambda a, b: a + b, _gen_add, n=2000)
    stress(add_32, lambda a, b: a + b, lambda r: (r.randint(-20, 20), r.randint(-20, 20)), n=500, seed=1)

ok  add_32: 6 cases
ok  add_32 vs brute force: 2000 random cases
ok  add_32 vs brute force: 500 random cases


## 2. Complexity, measured
**Amortized growth of a Python list.** `append` is amortized $O(1)$ because CPython over-allocates in proportion to
the size (about 1/8 extra, `list_resize` in `Objects/listobject.c`). A geometric growth factor means only $O(\log n)$
reallocations, and the total copying is $O(n)$. Below we watch `sys.getsizeof` change while appending $10^5$ items.

In [8]:
PTR = struct.calcsize("P")
a, base = [], sys.getsizeof([])
caps = []
for i in range(100_000):
    a.append(i)
    cap = (sys.getsizeof(a) - base) // PTR
    if not caps or cap != caps[-1]:
        caps.append(cap)
ratios = [caps[i + 1] / caps[i] for i in range(len(caps) - 5, len(caps) - 1)]
print(f"{len(caps)} reallocations for 100000 appends; first capacities {caps[:10]}; late growth ratios {[round(x, 3) for x in ratios]}")
print(f"total elements copied by reallocations <= {sum(caps[:-1])} (= {sum(caps[:-1]) / len(a):.1f} per append)")
check("list growth is geometric: O(log n) reallocations", len(caps) < 200 and min(ratios) > 1.05)

66 reallocations for 100000 appends; first capacities [4, 8, 16, 24, 32, 40, 52, 64, 76, 92]; late growth ratios [1.125, 1.125, 1.125, 1.125]
total elements copied by reallocations <= 798128 (= 8.0 per append)
ok  list growth is geometric: O(log n) reallocations


**Template: estimate the exponent from timings.** If $t \approx c\,n^k$, then $\log t = \log c + k\log n$. Fill in the
least-squares slope of $\log t$ against $\log n$. It is the quickest empirical check that your "O(n log n)" is not
secretly $O(n^2)$.

In [9]:
def loglog_slope(ns, ts):
    xs, ys = [math.log(n) for n in ns], [math.log(t) for t in ts]
    mx, my = sum(xs) / len(xs), sum(ys) / len(ys)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / sum((x - mx) ** 2 for x in xs)


ns_ = [1000, 2000, 4000, 8000, 16000]
check("slope of c*n^2 is 2", abs(loglog_slope(ns_, [3e-9 * n**2 for n in ns_]) - 2) < 1e-9)
check("slope of c*n is 1", abs(loglog_slope(ns_, [5e-7 * n for n in ns_]) - 1) < 1e-9)
check("slope of n log n is a bit above 1", 1.05 < loglog_slope(ns_, [n * math.log(n) for n in ns_]) < 1.2)


def drain_list(n):                  # list used as a queue: pop(0) shifts everything, O(n) each
    q = list(range(n))
    while q:
        q.pop(0)


def drain_deque(n):                 # deque.popleft is O(1)
    q = deque(range(n))
    while q:
        q.popleft()


def best_time(f, n, reps=3):
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        f(n)
        best = min(best, time.perf_counter() - t0)
    return best


ns_q = [5_000, 10_000, 20_000, 40_000]
for f in (drain_list, drain_deque):
    ts = [best_time(f, n) for n in ns_q]
    print(f"{f.__name__:12s} times {[f'{t * 1e3:.1f}ms' for t in ts]}  log-log slope {loglog_slope(ns_q, ts):.2f}")
print("(expect a slope toward 2 for list.pop(0) at large n, since memmove is fast and the quadratic term "
      "shows only for big n, and about 1 for deque.popleft; timings are machine dependent, so nothing is asserted)")


def py_loop(n):                      # one interpreted add per iteration
    x = 0
    for i in range(n):
        x += i
    return x


t_loop = best_time(py_loop, 1_000_000)
print(f"this machine: about {1e6 / t_loop / 1e6:.0f} million simple Python loop iterations per second")

ok  slope of c*n^2 is 2
ok  slope of c*n is 1
ok  slope of n log n is a bit above 1


drain_list   times ['1.6ms', '7.0ms', '27.9ms', '111.6ms']  log-log slope 2.03
drain_deque  times ['0.3ms', '0.5ms', '0.6ms', '1.3ms']  log-log slope 0.71
(expect a slope toward 2 for list.pop(0) at large n, since memmove is fast and the quadratic term shows only for big n, and about 1 for deque.popleft; timings are machine dependent, so nothing is asserted)
this machine: about 32 million simple Python loop iterations per second


## 3. Complexity self-quiz (answer out loud, then check the solution notebook)
Give tight $\Theta$ bounds and say *why* (recursion tree, master theorem case, aggregate argument).
1. $T(n) = 2T(n/2) + \Theta(n)$. Which algorithm?
2. $T(n) = T(n/2) + \Theta(1)$, and $T(n) = 2T(n/2) + \Theta(1)$.
3. $T(n) = 3T(n/2) + \Theta(n)$, and $T(n) = 4T(n/2) + \Theta(n^2)$.
4. $T(n) = T(n-1) + \Theta(n)$, and $T(n) = 2T(n-1) + \Theta(1)$. Why does the master theorem not apply?
5. Naive recursive Fibonacci: how many calls? With `@cache`?
6. `for i in range(1, n + 1): for j in range(0, n, i): ...`: total iterations?
7. `i = 1; while i * i <= n: i += 1` and `while n > 0: n //= 2`: number of iterations?
8. A dynamic array doubles its capacity when full. Total cost of $n$ appends (aggregate method)? What if it grew by a
   constant 100 slots each time instead?
9. A binary counter with $k$ bits is incremented $n$ times. Worst case of one increment vs amortized cost?
10. `def is_pal(s): return len(s) < 2 or (s[0] == s[-1] and is_pal(s[1:-1]))`: time and space? How do you fix it?
11. Building a string with `res += ch` in a loop vs `"".join(parts)`; `x in lst` inside a loop over `lst`;
    `sorted()` inside a loop. Name the hidden costs.
12. When is an $O(n^2)$ algorithm acceptable in an interview? Use a rough operations-per-second rule for Python.

<details><summary>Model answers</summary>

Master theorem (CLRS Ch. 4; Wikipedia): for $T(n) = aT(n/b) + f(n)$, compare $f(n)$ with $n^{\log_b a}$. If $f$ is
polynomially smaller, $T = \Theta(n^{\log_b a})$. If they are equal up to $\log^k$, $T = \Theta(n^{\log_b a}\log^{k+1} n)$.
If $f$ is polynomially larger (and regular), $T = \Theta(f(n))$.

1. $\Theta(n\log n)$, case 2 ($n^{\log_2 2} = n = f$). The recursion tree has $\log n$ levels of $\Theta(n)$ work.
   Merge sort.
2. $\Theta(\log n)$ (binary search, case 2 with $f = 1 = n^0$). $\Theta(n)$ (tree traversal, case 1: $n^{\log_2 2} = n$
   dominates $1$).
3. $\Theta(n^{\log_2 3}) \approx \Theta(n^{1.585})$ (Karatsuba, case 1). $\Theta(n^2\log n)$ (case 2, $n^{\log_2 4} = n^2$).
4. $\Theta(n^2)$, since $\sum_{i \le n} i$ (e.g. insertion sort's worst case, or quicksort with bad pivots). $\Theta(2^n)$
   (Towers of Hanoi). The subproblem is $n-1$, not $n/b$, so the master theorem's form does not apply. Unroll the recurrence instead.
5. $T(n) = T(n-1) + T(n-2) + 1$ gives $\Theta(\varphi^n)$ calls, $\varphi = (1+\sqrt5)/2 \approx 1.618$. Memoized: $n+1$
   distinct states, each computed once, $\Theta(n)$ time, but recursion depth $n$ (see A1.9 on CPython's limit of 1000).
6. $\sum_{i=1}^{n} \lceil n/i\rceil = \Theta(n\log n)$ (harmonic series). This is the same argument as the sieve of
   Eratosthenes, which over primes only is $\Theta(n\log\log n)$.
7. $\Theta(\sqrt n)$ and $\Theta(\log n)$ (precisely $\lfloor\log_2 n\rfloor + 1$ iterations for $n \ge 1$).
8. Doubling: copies cost $1 + 2 + 4 + \dots + 2^{\lfloor\log n\rfloor} < 2n$, so the total is $\Theta(n)$ and each
   append is amortized $O(1)$ (CLRS Ch. 16, dynamic tables). Constant growth by 100: about $n/100$ reallocations
   copying $100, 200, \dots$, a total of $\Theta(n^2/100) = \Theta(n^2)$, so amortized $\Theta(n)$ per append. Any
   **geometric** factor (CPython uses about 1.125, measured above) keeps $O(1)$ amortized.
9. Worst case $\Theta(k)$ (`0111...1` becomes `1000...0`). Amortized $O(1)$: bit $i$ flips every $2^i$ increments,
   so the total is $\sum_i n/2^i < 2n$ (aggregate). The accounting method charges 2 per increment, 1 to set a bit and
   1 prepaid for clearing it later (CLRS Ch. 16; Tarjan 1985).
10. Each slice `s[1:-1]` copies, so $\Theta(n^2)$ time. The $n/2$ live frames each hold their own slice, so the peak
    space is $\Theta(n^2)$, and the depth also hits the recursion limit for long strings. Fix: pass indices `i, j` (or loop with two pointers), giving $\Theta(n)$ time and $O(1)$ space.
11. Repeated `+=` on `str` can copy the prefix each time, $\Theta(n^2)$ worst case (CPython sometimes optimizes it in
    place; do not rely on that). `"".join` is $\Theta(n)$. `x in lst` is a linear scan, so the loop is $\Theta(n^2)$:
    use a `set`. `sorted()` inside a loop costs $\Theta(n\log n)$ per iteration: sort once, or use a heap or bisect
    (Python wiki TimeComplexity).
12. Rule of thumb: CPython runs on the order of $10^7$ simple interpreted operations per second (measured above for
    this machine; judges and laptops vary by a few times). So $n \le$ a few
    thousand allows $O(n^2)$, $n \approx 10^5$-$10^6$ needs $O(n\log n)$ or $O(n)$, and $n \le 20$ allows
    $O(2^n n)$. Always state the brute force and its complexity, then say whether the constraints require something better.
</details>

<!-- practice:start -->
## Practice list (solve these on LeetCode)

NeetCode 150 problems for this topic, generated from `neetcode-gh/leetcode@d5ebb3e` `.problemSiteData.json` (https://neetcode.io/practice). **B75** = also in Blind 75. Tick them off in your session copy; the full checklist is `A_coding/A1_problem_list.md`. Grind 75 (https://www.techinterviewhandbook.org/grind75/) is an alternative ordering of a similar set.

**Bit Manipulation** · plan week 16 · 7 problems

- [ ] [Single Number](https://leetcode.com/problems/single-number/) · Easy
- [ ] [Number of 1 Bits](https://leetcode.com/problems/number-of-1-bits/) · Easy · **B75**
- [ ] [Counting Bits](https://leetcode.com/problems/counting-bits/) · Easy · **B75**
- [ ] [Reverse Bits](https://leetcode.com/problems/reverse-bits/) · Easy · **B75**
- [ ] [Missing Number](https://leetcode.com/problems/missing-number/) · Easy · **B75**
- [ ] [Sum of Two Integers](https://leetcode.com/problems/sum-of-two-integers/) · Medium · **B75**
- [ ] [Reverse Integer](https://leetcode.com/problems/reverse-integer/) · Medium

**Math & Geometry** · plan week 16 · 8 problems

- [ ] [Rotate Image](https://leetcode.com/problems/rotate-image/) · Medium · **B75**
- [ ] [Spiral Matrix](https://leetcode.com/problems/spiral-matrix/) · Medium · **B75**
- [ ] [Set Matrix Zeroes](https://leetcode.com/problems/set-matrix-zeroes/) · Medium · **B75**
- [ ] [Happy Number](https://leetcode.com/problems/happy-number/) · Easy
- [ ] [Plus One](https://leetcode.com/problems/plus-one/) · Easy
- [ ] [Pow(x, n)](https://leetcode.com/problems/powx-n/) · Medium
- [ ] [Multiply Strings](https://leetcode.com/problems/multiply-strings/) · Medium
- [ ] [Detect Squares](https://leetcode.com/problems/detect-squares/) · Medium
<!-- practice:end -->

In [10]:
s.finish(lookups=0, score=None, notes="")

[A1.10] done in 0.0 min (within target 60 min)


0.013208520412445069